# Student Feedback Analysis Using Natural Language Processing

**Academic NLP demonstration.** The dataset is synthetically generated and does not represent real student opinions.

## Objectives
Demonstrate dataset inspection, preprocessing, tokenization, stopword removal, lemmatization, Bag of Words, TF-IDF, classification, evaluation, word analysis, and predictions.

## 1. Import Libraries and Load Dataset
Run this notebook from the project root or from the `notebooks` folder.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve(); ROOT = ROOT if (ROOT / 'src').exists() else ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from src.data_loader import load_dataset, dataset_summary
from src.preprocessing import ensure_nltk_resources, clean_text, tokenize, preprocess_text
from src.feature_extraction import make_bow_vectorizer, make_tfidf_vectorizer
from src.model import make_logistic_regression, make_naive_bayes
from src.evaluation import evaluate_predictions, comparison_table
from src.visualization import plot_sentiment_distribution, plot_word_frequency, plot_wordclouds, plot_confusion_matrix
ensure_nltk_resources(download=True)
data = load_dataset(ROOT / 'Student_Feedback_NLP.csv')
data.head()

## 2. Dataset Inspection, Missing Values, Duplicate Analysis, and Sentiment Distribution

In [ ]:
print(dataset_summary(data))
display(data.isna().sum().to_frame('Missing values'))
print('Duplicate records:', data.duplicated().sum())
display(data['Sentiment'].value_counts().rename_axis('Sentiment').to_frame('Count'))
plot_sentiment_distribution(data, ROOT / 'outputs' / 'sentiment_distribution.png')

## 3. Text Preprocessing
The pipeline lowercases, removes punctuation/special characters, tokenizes, removes English stopwords, and lemmatizes words. Lemmatization is preferred over stemming because it produces readable base forms.

In [ ]:
example = data.loc[0, 'Feedback']
print('Raw:', example)
print('Cleaned:', clean_text(example))
print('Tokens:', tokenize(example))
print('Processed:', preprocess_text(example))
data = data.copy()
data['Processed_Feedback'] = data['Feedback'].fillna('').map(preprocess_text)
data = data[data['Processed_Feedback'].str.len() > 0]
data[['Feedback', 'Processed_Feedback', 'Sentiment']].head()

## 4. Bag of Words and TF-IDF
Bag of Words counts vocabulary occurrences. TF-IDF is the primary representation because it downweights terms common throughout the corpus.

In [ ]:
bow = make_bow_vectorizer()
bow_matrix = bow.fit_transform(data['Processed_Feedback'])
print('Bag-of-Words shape:', bow_matrix.shape)
x_train, x_test, y_train, y_test = train_test_split(data['Processed_Feedback'], data['Sentiment'], test_size=0.2, random_state=42, stratify=data['Sentiment'])
tfidf = make_tfidf_vectorizer()
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)
print('TF-IDF train/test shapes:', x_train_tfidf.shape, x_test_tfidf.shape)

## 5. Logistic Regression and Multinomial Naive Bayes
The train/test split is stratified. The vectorizer is fitted only on training feedback, avoiding data leakage.

In [ ]:
models = {'Logistic Regression': make_logistic_regression(), 'Multinomial Naive Bayes': make_naive_bayes()}
results = {}
for name, model in models.items():
    model.fit(x_train_tfidf, y_train)
    prediction = model.predict(x_test_tfidf)
    results[name] = evaluate_predictions(y_test, prediction)
    print('\n', name, '\n', results[name]['report'])
comparison = comparison_table(results)
display(comparison)

## 6. Evaluation and Confusion Matrix
Accuracy is overall correctness. Precision, recall, and F1 are reported as weighted averages and class-wise in the classification reports.

In [ ]:
best_name = comparison.iloc[0]['Model']
print('Best model:', best_name)
plot_confusion_matrix(results[best_name]['confusion_matrix'], ROOT / 'outputs' / 'confusion_matrix.png')
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'outputs' / 'confusion_matrix.png')))

## 7. Word Frequency and Word Clouds
The chart lists frequent processed words. The word-cloud figure separates Positive, Neutral, and Negative vocabulary.

In [ ]:
plot_word_frequency(data['Processed_Feedback'], ROOT / 'outputs' / 'word_frequency.png')
plot_wordclouds(data, ROOT / 'outputs' / 'wordcloud.png')
display(Image(filename=str(ROOT / 'outputs' / 'word_frequency.png')))
display(Image(filename=str(ROOT / 'outputs' / 'wordcloud.png')))

## 8. Example Predictions, Results, and Conclusion
The final system uses the best evaluated model. Results on this synthetic dataset are for academic demonstration only; future work includes real consented feedback, aspect sentiment, multilingual processing, and transformers.

In [ ]:
best_model = models[best_name]
examples = ['The practical sessions are very helpful and I understand the concepts well.', 'The course is manageable but some topics need more explanation.', 'I am struggling to understand the confusing concepts.']
for text in examples:
    print(text, '→', best_model.predict(tfidf.transform([preprocess_text(text)]))[0])